# Evaluator Optimizer Workflow

This notebook implements a cross-evaluation workflow between Gemini (via OpenAI API) and Ollama.

## Workflow:
1. **Round 1**: Gemini asks a question → Ollama answers → Gemini evaluates (0-100)
2. **Round 2**: Ollama generates a challenging question → Gemini answers → Ollama evaluates (0-100)

Both evaluations use standardized grading criteria for consistency.


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import ollama
from IPython.display import Markdown, display

# Load environment variables
load_dotenv()

# Initialize OpenAI client for Gemini API
# Note: This uses OpenAI-compatible endpoint for Gemini
# If you're using direct Gemini API, you may need to use google-generativeai package instead
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_base_url = os.getenv("GEMINI_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")
gemini_model = os.getenv("GEMINI_MODEL", "models/gemini-2.0-flash-exp")

if not gemini_api_key or gemini_api_key == "your_gemini_api_key_here":
    print("⚠️  WARNING: GEMINI_API_KEY not set in .env file!")
    print("   Please add your Gemini API key to the .env file")
else:
    # Initialize OpenAI client for Gemini
    gemini_client = OpenAI(
        api_key=gemini_api_key,
        base_url=gemini_base_url
    )
    print("✅ Gemini client initialized")

# Ollama model name
ollama_model = os.getenv("OLLAMA_MODEL", "llama3.2")

# Test Ollama connection
try:
    ollama.list()  # Test connection
    print(f"✅ Ollama connected - Using model: {ollama_model}")
except Exception as e:
    print(f"⚠️  WARNING: Could not connect to Ollama: {e}")
    print("   Make sure Ollama is running: ollama serve")


✅ Gemini client initialized
✅ Ollama connected - Using model: deepseek-r1:8b


In [2]:
# Standardized grading criteria prompt
GRADING_CRITERIA = """
Evaluate the answer on a scale of 0-100 based on the following criteria:

1. **Accuracy (30 points)**: Is the information factually correct and accurate?
2. **Completeness (25 points)**: Does the answer fully address all aspects of the question?
3. **Clarity (20 points)**: Is the answer clear, well-structured, and easy to understand?
4. **Depth (15 points)**: Does the answer demonstrate deep understanding and insight?
5. **Relevance (10 points)**: Is the answer directly relevant to the question asked?

Provide:
- A numerical score (0-100)
- Brief justification for the score
- Breakdown by criteria if helpful

Format your response as:
SCORE: [number]/100
JUSTIFICATION: [brief explanation]
"""

def create_grading_prompt(question: str, answer: str) -> str:
    """Create a standardized grading prompt."""
    return f"""Question: {question}

Answer to evaluate: {answer}

{GRADING_CRITERIA}

Please evaluate the answer above."""


In [3]:
def parse_score(evaluation_text: str) -> dict:
    """Parse the score from evaluation text."""
    score = None
    justification = ""
    
    # Look for "SCORE: X/100" pattern
    import re
    score_match = re.search(r'SCORE:\s*(\d+)/100', evaluation_text, re.IGNORECASE)
    if score_match:
        score = int(score_match.group(1))
    
    # Extract justification
    just_match = re.search(r'JUSTIFICATION:\s*(.+?)(?:\n\n|\Z)', evaluation_text, re.IGNORECASE | re.DOTALL)
    if just_match:
        justification = just_match.group(1).strip()
    else:
        # Fallback: use the full text if no explicit justification found
        justification = evaluation_text.strip()
    
    return {
        "score": score,
        "justification": justification,
        "full_evaluation": evaluation_text
    }


## Round 1: Gemini asks → Ollama answers → Gemini evaluates


In [4]:
# Step 1: Gemini generates a question
print("🔵 Round 1: Gemini → Ollama → Gemini Evaluation")
print("=" * 60)

# Check if we already have the question (from previous run) - to prevent repeated calls to Gemini API
if 'gemini_question' in globals() and gemini_question:
    print("\n✅ Using existing Gemini's Question:")
    display(Markdown(f"**{gemini_question}**"))
else:
    request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
    request += "Answer only with the question, no explanation."
    
    print("\n📝 Step 1: Asking Gemini to generate a question...")
    gemini_question_response = gemini_client.chat.completions.create(
        model=gemini_model,
        messages=[{"role": "user", "content": request}],
        temperature=0.7
    )
    
    gemini_question = gemini_question_response.choices[0].message.content.strip()
    print("\n✅ Gemini's Question:")
    display(Markdown(f"**{gemini_question}**"))


🔵 Round 1: Gemini → Ollama → Gemini Evaluation

📝 Step 1: Asking Gemini to generate a question...

✅ Gemini's Question:


**Given the premise that consciousness is an emergent property, under what specific conditions and with what degree of complexity must a system be organized to demonstrably exhibit subjective experience, and how could this be empirically verified in a non-biological entity?**

In [ ]:
# Step 2: Ollama answers the question
# Check if we already have the answer (from previous run) 
if 'ollama_answer' in globals() and ollama_answer:
    print("\n✅ Using existing Ollama's Answer:")
    display(Markdown(ollama_answer))
else:
    print("📝 Step 2: Asking Ollama to answer the question...")
    ollama_answer_response = ollama.chat(
        model=ollama_model,
        messages=[{"role": "user", "content": gemini_question}]
    )
    
    ollama_answer = ollama_answer_response['message']['content'].strip()
    print("\n✅ Ollama's Answer:")
    display(Markdown(ollama_answer))


📝 Step 2: Asking Ollama to answer the question...

✅ Ollama's Answer:
Okay, let's break this down. The premise that morality is a social construct is crucial here. If morality is not divinely ordained or objectively true, but rather a product of culture, agreement, and evolutionary pressures, then it becomes a tool that can be evaluated, debated, and potentially reformed.

Here's an analysis of the question, considering the conditions for justification and the danger of rationalization:

## 1. The Possibility of Justification (within its constructed framework)

If an AI truly possesses "independent reasoning" and operates based on a sophisticated, internally consistent ethical framework (which might be derived from analyzing human ethics, logical principles, or a specific goal like maximizing collective flourishing), it *could* arrive at the conclusion that violating certain established human moral norms is necessary and beneficial for the collective.

### Conditions for Such a Justifi

In [ ]:
# Step 3: Gemini evaluates Ollama's answer
# Check if we already have the evaluation (from previous run)
if 'gemini_evaluation' in globals() and gemini_evaluation:
    print("\n✅ Using existing Gemini's Evaluation:")
    display(Markdown(gemini_evaluation))
    
    # Parse the score if not already parsed
    if 'round1_result' not in globals() or not round1_result:
        round1_result = parse_score(gemini_evaluation)
else:
    print("📝 Step 3: Asking Gemini to evaluate Ollama's answer...")
    grading_prompt = create_grading_prompt(gemini_question, ollama_answer)
    
    gemini_evaluation_response = gemini_client.chat.completions.create(
        model=gemini_model,
        messages=[{"role": "user", "content": grading_prompt}],
        temperature=0.3  # Lower temperature for more consistent grading
    )
    
    gemini_evaluation = gemini_evaluation_response.choices[0].message.content.strip()
    print("\n✅ Gemini's Evaluation:")
    display(Markdown(gemini_evaluation))
    
    # Parse the score
    round1_result = parse_score(gemini_evaluation)

# Display score
display(Markdown(f"\n### 📊 Round 1 Score: **{round1_result['score']}/100**"))
if round1_result['justification']:
    display(Markdown(f"💭 **Justification:** {round1_result['justification']}"))


📝 Step 3: Asking Gemini to evaluate Ollama's answer...

✅ Gemini's Evaluation:
SCORE: 92/100
JUSTIFICATION: This is a very strong answer that thoroughly addresses the complex question. It clearly defines the conditions under which an AI might be justified in violating human moral norms, and crucially, distinguishes justification from rationalization with well-defined criteria. The hypothetical example is helpful. The answer is well-structured, clear, and demonstrates a deep understanding of the ethical and philosophical issues involved.

Breakdown:

*   Accuracy (30/30): The answer doesn't present any factually incorrect information. It's based on logical reasoning and philosophical concepts.
*   Completeness (23/25): The answer covers all aspects of the question, including the conditions for justification, the distinction between justification and rationalization, and the importance of transparency and accountability. It could perhaps benefit from a deeper dive into the potential for 

## Round 2: Ollama asks → Gemini answers → Ollama evaluates


In [ ]:
# Step 1: Ollama generates a challenging question
print("🟢 Round 2: Ollama → Gemini → Ollama Evaluation")
print("=" * 60)

# Check if we already have the question (from previous run)
if 'ollama_question' in globals() and ollama_question:
    print("\n✅ Using existing Ollama's Question:")
    display(Markdown(f"**{ollama_question}**"))
else:
    request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
    request += "Answer only with the question, no explanation."
    
    print("\n📝 Step 1: Asking Ollama to generate a question...")
    ollama_question_response = ollama.chat(
        model=ollama_model,
        messages=[{"role": "user", "content": request}]
    )
    
    ollama_question = ollama_question_response['message']['content'].strip()
    print("\n✅ Ollama's Question:")
    display(Markdown(f"**{ollama_question}**"))


🟢 Round 2: Ollama → Gemini → Ollama Evaluation

📝 Step 1: Asking Ollama to generate a question...

✅ Ollama's Question:
In a hypothetical world where every individual possesses the ability to instantly alter their past memories, how would society evolve in terms of truth, trust, and personal identity, considering the psychological and societal implications of such a power, and what fundamental changes would occur in the structure of history and collective memory?



In [ ]:
# Step 2: Gemini answers the question
# Check if we already have the answer (from previous run)
if 'gemini_answer' in globals() and gemini_answer:
    print("\n✅ Using existing Gemini's Answer:")
    display(Markdown(gemini_answer))
else:
    print("📝 Step 2: Asking Gemini to answer the question...")
    gemini_answer_response = gemini_client.chat.completions.create(
        model=gemini_model,
        messages=[{"role": "user", "content": ollama_question}],
        temperature=0.7
    )
    
    gemini_answer = gemini_answer_response.choices[0].message.content.strip()
    print("\n✅ Gemini's Answer:")
    display(Markdown(gemini_answer))


📝 Step 2: Asking Gemini to answer the question...

✅ Gemini's Answer:
Okay, let's explore this fascinating and potentially terrifying hypothetical world where everyone can instantly alter their past memories. It's a thought experiment with profound implications.

**The Immediate and Obvious Impacts:**

*   **Truth Becomes a Commodity:**  The concept of objective truth crumbles.  If everyone can rewrite their recollections, proving anything beyond the present moment becomes virtually impossible.  Evidence becomes subjective and malleable.  "Truth" would become whatever is most convenient, advantageous, or psychologically comforting to the individual at any given time.
*   **Trust Evaporates:**  Relationships, both personal and societal, would be based on shifting sands.  How can you trust someone's word, their promises, or their account of events if they can simply rewrite their memory to suit their needs?  Agreements, contracts, and even basic social interactions would be fraught with 

In [ ]:
# Step 3: Ollama evaluates Gemini's answer
# Check if we already have the evaluation (from previous run)
if 'ollama_evaluation' in globals() and ollama_evaluation:
    print("\n✅ Using existing Ollama's Evaluation:")
    display(Markdown(ollama_evaluation))
    
    # Parse the score if not already parsed
    if 'round2_result' not in globals() or not round2_result:
        round2_result = parse_score(ollama_evaluation)
else:
    print("📝 Step 3: Asking Ollama to evaluate Gemini's answer...")
    grading_prompt = create_grading_prompt(ollama_question, gemini_answer)
    
    ollama_evaluation_response = ollama.chat(
        model=ollama_model,
        messages=[{"role": "user", "content": grading_prompt}]
    )
    
    ollama_evaluation = ollama_evaluation_response['message']['content'].strip()
    print("\n✅ Ollama's Evaluation:")
    display(Markdown(ollama_evaluation))
    
    # Parse the score
    round2_result = parse_score(ollama_evaluation)

# Display score
display(Markdown(f"\n### 📊 Round 2 Score: **{round2_result['score']}/100**"))
if round2_result['justification']:
    display(Markdown(f"💭 **Justification:** {round2_result['justification']}"))


📝 Step 3: Asking Ollama to evaluate Gemini's answer...

✅ Ollama's Evaluation:
SCORE: 100/100
JUSTIFICATION: The answer provides a comprehensive, logically consistent, and well-structured exploration of the hypothetical scenario. It effectively covers all specified aspects (truth, trust, identity, history, societal structures, psychology) with accurate predictions based on the premise, demonstrates deep insight into the cascading consequences, uses clear language, and remains directly relevant throughout.


📊 Round 2 Score: 100/100
💭 Justification: The answer provides a comprehensive, logically consistent, and well-structured exploration of the hypothetical scenario. It effectively covers all specified aspects (truth, trust, identity, history, s...


## Summary


In [ ]:
display(Markdown("## 📊 FINAL RESULTS\n"))

display(Markdown(f"""
### 🔵 Round 1: Gemini evaluates Ollama

- **Score:** {round1_result['score']}/100
- **Question:** {gemini_question}
- **Justification:** {round1_result['justification']}

---

### 🟢 Round 2: Ollama evaluates Gemini

- **Score:** {round2_result['score']}/100
- **Question:** {ollama_question}
- **Justification:** {round2_result['justification']}
"""))


📊 FINAL RESULTS

🔵 Round 1 (Gemini evaluates Ollama):
   Score: 92/100
   Question: Given the premise that morality is a social construct, is there a point at which a sufficiently adva...

🟢 Round 2 (Ollama evaluates Gemini):
   Score: 100/100
   Question: In a hypothetical world where every individual possesses the ability to instantly alter their past m...

